# 并行实验：AFT 删失回归 (AFT Censored Regression)

**评估指标说明：**
根据论文公式 (31)，本实验的核心评估指标是 **系数的 RMSE（即 L2 范数距离）**：
$$ \text{RMSE} = \frac{1}{S} \sum_{s=1}^S \sqrt{\sum_{l=1}^p (\hat{\theta}_{l,s} - \theta_l^*)^2} $$
我们在代码中严格遵循了这一指标。

In [ ]:
import os
import json
import time
import numpy as np
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt
from multiprocessing import Pool
from utils.runner import run_single_aft


In [ ]:
# ==========================================
# 参数设定 (Hyperparameters)
# ==========================================
NUM_RUNS = 200       # 重复实验次数
NUM_WORKERS = 10     # 进程数

import numpy as np

params = {
    "Experiment": "AFT Survival",
    "m": 10, 
    "n": 100,
    "p_prime": 5, 
    "p": 5, 
    "pc": 0.3,
    "T": 40, 
    "W_inner": 5, 
    "rho": 1.3, 
    "ic_type": "bic", 
    "lambda_candidates": np.logspace(-2.5, -1.5, 10).tolist(),
    "noise_type": "t1",
    "rng_seed": 245,
    "run_proposed": True,
    "run_local": True,
    "run_baselines": False 
}

import os
folder = "ranking" if "Ranking" in params["Experiment"] else "aft"
os.makedirs(folder, exist_ok=True)
filename = f"{folder}/exp_{folder}_{NUM_RUNS}_{params['noise_type']}_p{params['p']}_pc_{str(params['pc']).replace('.', '')}_rho_{str(params['rho']).replace('.', '')}.json"

print(f"Starting Parallel Experiments: {NUM_RUNS} runs...")
results = []
def update_progress(result):
    results.append(result)
    print(f"\rProgress: {len(results)}/{NUM_RUNS} runs completed.", end="", flush=True)

def log_error(e):
    print(f"\nError in worker process: {e}")

from multiprocessing import Pool
from utils.runner import run_single_ranking, run_single_aft

runner_fn = run_single_ranking if "Ranking" in params["Experiment"] else run_single_aft

if __name__ == '__main__':
    with Pool(NUM_WORKERS) as pool:
        for i in range(NUM_RUNS):
            pool.apply_async(runner_fn, args=(i, params), callback=update_progress, error_callback=log_error)
        pool.close()
        pool.join()
    print() 

    with open(filename, 'w', encoding='utf-8') as f:
        json.dump({'parameters': params, 'results': results}, f, ensure_ascii=False, indent=4)
    print(f"Results saved to {filename}")




In [ ]:
# ==========================================
# 实验完成
# ==========================================
print(f"\n实验完成！数据已保存至 {filename}")
print("请打开 exp2_plot_aft.ipynb 读取该文件以生成 Markdown 表格和收敛曲线。\n")
